# Demo RAG Assistant — Knowledge Graph Sportifs & Competitions

Ce notebook illustre le pipeline complet :
1. Chargement de la KB
2. Detection d'entites et generation SPARQL
3. Execution de la requete sur la KB locale
4. Generation de la reponse via LLM (ou mode simulation)

In [ ]:
import sys
from pathlib import Path

# Ajouter le dossier racine du projet au path
PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Racine du projet : {PROJECT_ROOT}')

In [ ]:
from rag.query_builder import QueryBuilder, build_query

# Test build_query (interface simple)
print('=== Interface build_query ===' )
for intent in ['medals', 'competitions', 'sport', 'country']:
    q = build_query(intent, 'UsainBolt')
    print(f'\n[{intent}]\n{q}')

In [ ]:
from rag.sparql_executor import SPARQLExecutor

executor = SPARQLExecutor()
print(f'KB loaded: {executor.kb_path}')

In [ ]:
# Test pipeline complet
question = 'Quelles medailles a remporte Usain Bolt ?'
qb = QueryBuilder()
result = qb.construire_requete(question)
print(f'Question : {question}')
print(f'Entite   : {result["entite"]}')
print(f'Requete  : {result["query"]}')

triplets, fallback = executor.executer_avec_fallback(result['query'], result['entite'])
print(f'Triplets trouves : {len(triplets)}')

contexte = executor.formater_contexte(triplets, result['entite'], fallback)
print(f'\nContexte :\n{contexte}')

In [ ]:
from rag.llm_client import LLMClient

llm = LLMClient()
print(f'Fournisseur actif : {llm.fournisseur}')

reponse = llm.generer(contexte, question)
print(f'\nReponse :\n{reponse}')

In [ ]:
# Questions de test par lots
from rag.rag_assistant import ask

questions = [
    'Quelles medailles a remporte Usain Bolt ?',
    'Quel sport pratique Lionel Messi ?',
    'Quels athletes representent la France ?',
]

for q in questions:
    rep = ask(q)
    print(f'Q: {q}')
    print(f'R: {rep[:200]}')
    print()